# 3. Modele typu Encoder

Modele Encoderowe w przeciwieństwie do modeli Decoder, nie generują (przewidują), kolejnych tokenów. Ich głównym celem jest przede wszystkim zrozumienie tekstu. Ich zadaniem jest budowanie najlepszej reprezentacji tekstu.

Najbardziej klasycznym przykładem jest rodzina modeli BERT oraz jej późniejsze warianty (np. RoBERTa, DistilBERT czy XLM-RoBERTa). Modele te analizują wejście w sposób dwukierunkowy (od lewej i od prawej). Dzięki temu są szczególnie skuteczne w zadaniach:
- klasyfikacji tekstu,
- analizie sentymentu,
- wyszukiwaniu semantycznym,
- budowaniu embeddingów zdań i dokumentów,
- zadaniach typu extractive QA.


W praktyce pomagają one udzielić odpowiedzi na pytanie "co znaczy ten tekst?".

## Wymagania:
Konto (w razie napotakania ograniczeń): [huggingface.co](https://huggingface.co/)

- `transformers` - modele, tokenizery i pipeline’y z Hugging Face
- `datasets` - gotowe zbiory danych i wygodne przetwarzanie danych
- `evaluate` - liczenie metryk jakości modeli
- `accelerate` - prostsze uruchamianie obliczeń na GPU/CPU
- `sentence-transformers` - tworzenie embeddingów zdań i semantic search
- `scikit-learn` - pomocnicze narzędzia do analizy wyników i podobieństwa


In [1]:
# Instalacja pakietów w Colabie
!pip -q install transformers datasets evaluate accelerate sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


## Ekosystem huggingface

W ekosystemie Hugging Face najczęściej pracujemy z trzema elementami:

- tokenizer - zamienia tekst na tokeny i liczby zrozumiałe dla modelu,

- model - wykonuje obliczenia na tych danych,

- pipeline - wygodna, gotowa nakładka, która upraszcza użycie modelu.

### Ręczne wczytywanie modelu
- `AutoTokenizer` wczytuje tokenizer odpowiedni dla wybranego modelu,

- `AutoModelForMaskedLM` wczytuje model przygotowany do zadania masked language modeling,

- `from_pretrained(...)` pobiera gotowy checkpoint.

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
text = "Transformers are very useful for [MASK]."
inputs = tokenizer(text, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101, 19081,  2024,  2200,  6179,  2005,   103,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}


To zwraca słownik z danymi wejściowymi, np.:

- input_ids - numery tokenów,

- attention_mask - informacja, które tokeny są prawdziwym tekstem.

Oprócz tego możemy zobaczyć same tokeny:

In [ ]:
tokens = tokenizer.tokenize(text)
print(tokens)

['transformers', 'are', 'very', 'useful', 'for', '[MASK]', '.']


### Pipeline

`pipeline` to najprostszy sposób pracy z modelem, bo:

* sam pobiera model i tokenizer,
* sam przygotowuje dane wejściowe,
* sam wykonuje inferencję,
* zwraca gotowy wynik.


In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="bert-base-uncased")
result = fill_mask("Alicja przecięła [MASK], przez co on pękł.")

print(result)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.11297150701284409, 'token': 3501, 'token_str': '##j', 'sequence': 'alicja przeciełaj, przez co on pekł.'}, {'score': 0.09314999729394913, 'token': 2243, 'token_str': '##k', 'sequence': 'alicja przeciełak, przez co on pekł.'}, {'score': 0.0769791454076767, 'token': 8337, 'token_str': '##ska', 'sequence': 'alicja przeciełaska, przez co on pekł.'}, {'score': 0.06656984239816666, 'token': 10344, 'token_str': '##wski', 'sequence': 'alicja przecieławski, przez co on pekł.'}, {'score': 0.05849342420697212, 'token': 3900, 'token_str': '##ja', 'sequence': 'alicja przeciełaja, przez co on pekł.'}]


## Import wymaganych bibliotek

In [ ]:
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

from datasets import load_dataset
import evaluate

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import confusion_matrix, classification_report

## Określenie urządzenia

In [ ]:
device = 0 if torch.cuda.is_available() else -1
print("CUDA available:", torch.cuda.is_available())
print("Device for pipeline:", device)

CUDA available: True
Device for pipeline: 0


## Maskowanie - uzupełnianie tekstu z wykorzystaniem modeli encoder

Modele oparte wyłącznie na architekturze Encodera z rodziny BERT, trenowane są między innymi przez Masked Language Modeling, który polega na ukrywaniu części słów w zdaniu i uczeniu modelu przewidywania brakujących tokenów na podstawie kontekstu.

Przykładowo, podczas trenowania modelu mając zdanie:

- Alicja ma nożyczki i przeciąła nim balona przez co on pękł.

maskowane są przykładowe słowa takie jak np, "balona". przez co model widzi.
- Alicja ma nożyczki i przecięła nim `[MASK]`, przez co on pękł.

Zadaniem modelu jest przewidzieć, jakie słowo powinno znaleźć się w miejscu `[MASK]` na podstawie pozostałych słów w zdaniu. Podczas tego treningu model bierze pod uwagę zarówno lewą, jak i prawą stronę maskowanego słowa. Gdyby analizował tylko lewą stronę, to dla fragmentu „Alicja ma nożyczki i przecięła nim …” równie dobrze mógłby przewidzieć słowa takie jak „pasek”, „kartkę” czy „sznurek”. Dopiero uwzględnienie dalszej części zdania — „przez co on pękł” — pozwala poprawnie przewidzieć słowo „balon”.

### Przykład i zadanie

Uruchom poniższy kod i porównaj wyniki dla:
- `bert-base-uncased`
- `roberta-base`
- `distilbert-base-uncased`
- `FacebookAI/xlm-roberta-base`

Następnie dopisz własne 3 zdania:
- jedno po angielsku,
- jedno po polsku,
- jedno domenowe (np. informatyka, medycyna, edukacja).

In [ ]:
def run_fill_mask(model_id, text, top_k=5):
    tokenizer = AutoTokenizer.from_pretrained(model_id) # Załadowanie tokenizera
    mask_token = tokenizer.mask_token # Określenie tokenu "maski"

    if mask_token is None:
        raise ValueError(f"Model {model_id} nie posiada zdefiniowanego tokenu maski.")

    pipe = pipeline("fill-mask", model=model_id, tokenizer=tokenizer, device=device)

    print(f"\nMODEL: {model_id}")
    print("TEXT:", text.replace("[MASK]", mask_token).replace("<mask>", mask_token))

    outputs = pipe(text.replace("[MASK]", mask_token).replace("<mask>", mask_token), top_k=top_k)

    for i, out in enumerate(outputs, 1):
        print(f"{i}. {out['sequence']} | score={out['score']:.4f}")

models = [
    "bert-base-uncased"
]

text_en = "Transformers are very useful for [MASK] tasks."

for model_id in models:
    run_fill_mask(model_id, text_en)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



MODEL: bert-base-uncased
TEXT: Transformers are very useful for [MASK] tasks.
1. transformers are very useful for many tasks. | score=0.1458
2. transformers are very useful for complex tasks. | score=0.0672
3. transformers are very useful for some tasks. | score=0.0485
4. transformers are very useful for engineering tasks. | score=0.0448
5. transformers are very useful for various tasks. | score=0.0356


### Pytania
1. Który model najlepiej radzi sobie z językiem angielskim?
2. Który model najlepiej radzi sobie z językiem polskim?
3. Czy różnice między modelami wynikają tylko z architektury, czy też z danych treningowych?

## Tokenizacja i subtokeny
Tokenizer pełni funkcję kluczowego interfejsu między tekstem naturalnym a modelem obliczeniowym, odpowiadając za segmentację danych na jednostki zwane tokenami lub subtokenami. Wybór konkretnej strategii podziału determinuje zarówno wydajność procesową, jak i zdolność modelu do rozumienia struktur językowych, dlatego musi być ściśle skorelowany z charakterystyką zadania oraz specyfiką lingwistyczną danych wejściowych. W scenariuszach operujących na standardowym słownictwie efektywna okazuje się tokenizacja na poziomie całych słów, która minimalizuje długość sekwencji wejściowej i ułatwia interpretację semantyczną. Jednak w przypadku języków o rozbudowanej morfologii i fleksji, a także w obliczu terminologii specjalistycznej lub błędów zapisu, niezbędna staje się tokenizacja subtokenowa. Agresywny podział na mniejsze komponenty pozwala modelowi na poprawną obsługę form, których nie napotkał w trakcie treningu, co znacząco zwiększa elastyczność i zasięg słownika przy zachowaniu jego stałego rozmiaru. Ostatecznie proces ten jest balansowaniem między krótszymi, bardziej zwięzłymi reprezentacjami a szczegółową dekompozycją, która choć wydłuża sekwencję, umożliwia precyzyjne przetwarzanie rzadkich i złożonych struktur językowych.

### Przykład i zadania
Przedewszystkim uruchom kod.

Wybierz 3 własne zdania:
- jedno krótkie po polsku,
- jedno długie po polsku,
- jedno zawierające nazwę własną, skrót lub termin techniczny.


In [ ]:
def inspect_tokenization(model_id, text):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokens = tokenizer.tokenize(text)
    ids = tokenizer(text)["input_ids"]
    print(f"\nMODEL: {model_id}")
    print("Tekst:", text)
    print("Liczba tokenów:", len(ids))
    print("Tokeny:", tokens[:60])

texts = [
    "Transformers changed natural language processing.",
    "Modele językowe dobrze działają na zadaniach klasyfikacji tekstu.",
    "Politechnika Warszawska prowadzi zajęcia z dużych modeli językowych.",
]

for t in texts:
    print("\n" + "="*100)
    for model_id in ["bert-base-uncased", "roberta-base", "FacebookAI/xlm-roberta-base"]:
        inspect_tokenization(model_id, t)



MODEL: bert-base-uncased
Tekst: Transformers changed natural language processing.
Liczba tokenów: 8
Tokeny: ['transformers', 'changed', 'natural', 'language', 'processing', '.']

MODEL: roberta-base
Tekst: Transformers changed natural language processing.
Liczba tokenów: 9
Tokeny: ['Transform', 'ers', 'Ġchanged', 'Ġnatural', 'Ġlanguage', 'Ġprocessing', '.']

MODEL: FacebookAI/xlm-roberta-base
Tekst: Transformers changed natural language processing.
Liczba tokenów: 11
Tokeny: ['▁Trans', 'former', 's', '▁changed', '▁natural', '▁language', '▁process', 'ing', '.']


MODEL: bert-base-uncased
Tekst: Modele językowe dobrze działają na zadaniach klasyfikacji tekstu.
Liczba tokenów: 31
Tokeny: ['model', '##e', 'je', '##zy', '##kow', '##e', 'do', '##br', '##ze', 'd', '##zia', '##ła', '##ja', 'na', 'za', '##dan', '##iac', '##h', 'k', '##las', '##y', '##fi', '##ka', '##c', '##ji', 'te', '##ks', '##tu', '.']

MODEL: roberta-base
Tekst: Modele językowe dobrze działają na zadaniach klasyfikacji tek

Odpowiedz:
1. Który tokenizer daje najmniej tokenów?
2. Czy polskie słowa częściej rozpadają się na subtokeny niż angielskie?
3. Jak to może wpływać na jakość modelu?

## Token classification - NER i odmiany

NER, jest zadaniem klasyfikacji tokenów, które polega na przypisaniu kategorii do konkretnych części zdania. Klasyczne modele NER rozpoznają np.:
- osoby,
- miejsca,
- organizacje,
- daty itd...

Przykładowo w zdaniu „Anna Nowak pracuje w Google w Warszawie.”

Model może wykryć:

- Anna Nowak → osoba

- Google → organizacja

- Warszawie → lokalizacja



### Zadanie
1. Stwórz listę 5 zdań w j. angielskim, w których występują:
- imie i nazwisko,
- organizacja,
- lokalizacja,
- data albo wydarzenie.
2. Zaimplementuj pipeline, w którym zastosujesz model [`dslim/bert-base-NER`](https://huggingface.co/dslim/bert-base-NER)
3. Sprawdź działanie i postaraj się "zepsuć" model
4. Wyszukaj polskie odpowiedniki


Przydatne: [dokumentacja](https://huggingface.co/transformers/v4.11.1/_modules/transformers/pipelines/token_classification.html)

## Embeddingi - przykład

Encoder może służyć nie tylko do klasyfikacji, ale też do tworzenia reprezentacji wektorowych tekstu. Przykładem takiego zadania jest stworzenie prostej wyszukiwarki semantycznej.

Przygotowana funkcja zamienia zapytanie tekstowe na wektor liczb, czyli numeryczną reprezentację jego znaczenia. Wektory dokumentów są przygotowane wcześniej w taki sam sposób. Następnie dla każdego dokumentu obliczane jest podobieństwo między jego wektorem a wektorem zapytania za pomocą podobieństwa cosinusowego.

$$
\text{cos_sim}(A, B) = \frac{A \cdot B}{|A||B|}
$$

gdzie:

* $A$ to wektor zapytania,
* $B$ to wektor dokumentu.

Im większa wartość tego wyniku, tym bardziej podobne znaczeniowo są do siebie zapytanie i dokument. Na końcu dokumenty są sortowane malejąco według uzyskanego wyniku i zwracane są te, które najlepiej pasują do zapytania.


In [ ]:
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

docs = [
    "Warunkiem zaliczenia laboratorium jest oddanie kompletnego notebooka.",
    "Student może poprawić kolokwium podczas konsultacji prowadzącego.",
    "Projekt końcowy należy oddać do 30 czerwca.",
    "Na zajęciach można pracować w parach.",
    "Konsultacje odbywają się w środy o godzinie 14:00.",
    "Na laboratoriach używamy środowiska Google Colab.",
]

doc_embeddings = embed_model.encode(docs)

def semantic_search(query, docs, doc_embeddings, top_k=3):
    query_emb = embed_model.encode([query])
    scores = cosine_similarity(query_emb, doc_embeddings)[0]
    ranking = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return ranking

queries = [
    "Kiedy można poprawić kolokwium?",
    "Do kiedy trzeba oddać projekt?",
    "W jakim środowisku wykonujemy zadania?",
]

for q in queries:
    print("\nQUERY:", q)
    for doc, score in semantic_search(q, docs, doc_embeddings, top_k=3):
        print(f"{score:.4f} | {doc}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



QUERY: Kiedy można poprawić kolokwium?
0.4799 | Student może poprawić kolokwium podczas konsultacji prowadzącego.
0.2338 | Projekt końcowy należy oddać do 30 czerwca.
0.1743 | Warunkiem zaliczenia laboratorium jest oddanie kompletnego notebooka.

QUERY: Do kiedy trzeba oddać projekt?
0.6482 | Projekt końcowy należy oddać do 30 czerwca.
0.3142 | Konsultacje odbywają się w środy o godzinie 14:00.
0.2222 | Student może poprawić kolokwium podczas konsultacji prowadzącego.

QUERY: W jakim środowisku wykonujemy zadania?
0.4252 | Na zajęciach można pracować w parach.
0.3604 | Na laboratoriach używamy środowiska Google Colab.
0.3152 | Warunkiem zaliczenia laboratorium jest oddanie kompletnego notebooka.


## Trenowanie modelu do klasyfikacji tekstu

### Określenie rozmiaru zbiorów

In [ ]:
small_train = 1000
small_val = 300

### Zbiór danych

Zbiór danych dotyczy klasyfikacji binarnej sentymentu zdania, rozpatruje on problem czy zdanie wejściowe ma wydźwięk pozytywny czy negatywny

In [ ]:
dataset = load_dataset("glue", "sst2")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [ ]:
dataset["train"]

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 67349
})

In [ ]:
dataset["train"][0]

{'sentence': 'hide new secretions from the parental units ',
 'label': 0,
 'idx': 0}

### Wczytanie tokenizera

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

### Tokenizacja danych wejściowych

In [ ]:
def tokenize_batch(batch):
    return tokenizer(batch["sentence"], truncation=True)

tokenized = dataset.map(tokenize_batch, batched=True)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [ ]:
tokenized = tokenized.remove_columns(["sentence", "idx"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

### Zbiory treningowy i walidacyjny

In [ ]:
train_ds = tokenized["train"].shuffle(seed=42).select(range(small_train))
val_ds = tokenized["validation"].shuffle(seed=42).select(range(small_val))


In [ ]:
train_ds[0]

{'labels': tensor(1),
 'input_ids': tensor([  101, 12555,  1010, 11951,  1999, 22092,  2066,  2137, 11345,  1998,
          2757,  1011,  2006,  1999,  2602,  1010,   102]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [ ]:
args = TrainingArguments(
    output_dir="encoder_cls_model",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

In [ ]:
trainer.evaluate()

{'eval_loss': 0.7078104615211487,
 'eval_model_preparation_time': 0.0034,
 'eval_accuracy': 0.4766666666666667,
 'eval_f1': 0.3227990970654628,
 'eval_runtime': 0.4968,
 'eval_samples_per_second': 603.887,
 'eval_steps_per_second': 38.246}

In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
1,0.491267,0.375006,0.003400,0.836667,0.836650


TrainOutput(global_step=63, training_loss=0.491267340523856, metrics={'train_runtime': 4.0705, 'train_samples_per_second': 245.668, 'train_steps_per_second': 15.477, 'total_flos': 8811151813728.0, 'train_loss': 0.491267340523856, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.3750061094760895,
 'eval_model_preparation_time': 0.0034,
 'eval_accuracy': 0.8366666666666667,
 'eval_f1': 0.8366503316998366,
 'eval_runtime': 0.4031,
 'eval_samples_per_second': 744.179,
 'eval_steps_per_second': 47.131,
 'epoch': 1.0}